# SHACL Rules — User Guide

`StarShaclValidator.apply_rules()` executes SHACL's rule-based inference (`sh:rule`, `sh:TripleRule`, `sh:SPARQLRule`) - a *fourth* processing mode alongside `validate()` (checks conformance, never mutates), `evaluate()` (computes `sh:values`-declared virtual properties on demand, never persists), and `extract_subgraph()` (pulls out exactly the triples a shape covers). Rules are different from all three: they **materialize new, real triples** into the data graph, computed from existing ones.

The vocabulary here - `sh:rule`/`sh:TripleRule`/`sh:SPARQLRule`/`sh:condition`, plus the newer `sh:layer`/`sh:runOnce`/`sh:sourceRule`/`sh:RuleSet` - currently lives in the **SHACL 1.2 Inference Rules** editor's draft (`https://w3c.github.io/data-shapes/shacl12-inference-rules/`; not yet published at a `/TR/` URL). It briefly lived inside "SHACL 1.2 SPARQL Extensions" §8 and, before that, in a now-retired standalone "SHACL 1.2 Rules" document - the vocabulary and behavior haven't changed across any of these moves, only the citation has (see `docs/shacl12-gap-matrix.md` for the full history if you need it). This guide always means the RDF-native `sh:rule` vocabulary starshacl actually executes - not "SPARQL Rule Language" (SRL), a separate human-authoring text syntax for a different, general-purpose Datalog-style rules language that starshacl deliberately doesn't parse (RDF triples are the real interchange form here, the same position this project takes on SHACL's own Compact Syntax).

This guide builds up the feature one concept at a time: basic rule types, execution semantics (fixpoint iteration, layering, rule sets), provenance, RDF 1.2 interplay, and the mutation behavior you need to know before relying on this in real code.

## How to run this notebook

1. `pip install "git+https://github.com/hidden-graph/starlayer.git"` (or install the three packages editable from a local checkout — see the main user guide).
2. Run cells from top to bottom — later sections reuse the running example data from earlier ones.

In [1]:
from starlayergraph import StarLayerGraph, Namespace
from starshacl import StarShaclValidator

EX = Namespace("http://example.org/")

## 1. Basic rules: `sh:TripleRule`

A `sh:TripleRule` builds one new triple per matching focus node, from three node-expression templates: `sh:subject`, `sh:predicate`, `sh:object`. `sh:this` refers to the current focus node - the simplest possible template.

Here, every `ex:Person` gets a new `ex:isPerson true` triple. `apply_rules()` returns a `RulesResult` - use `result.data_graph` for the rule-updated graph (see section 11 for exactly what happens to the graph object you passed in).

**Example 1**

In [2]:
data = StarLayerGraph()
data.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:alice a ex:Person .
    ex:bob a ex:Person .
""", format="turtle12")

shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    ex:PersonRule a sh:NodeShape ; sh:targetClass ex:Person ;
      sh:rule [
        a sh:TripleRule ;
        sh:subject sh:this ; sh:predicate ex:isPerson ; sh:object true ;
      ] .
""", format="turtle12")

result = StarShaclValidator().apply_rules(data_graph=data, shacl_graph=shapes)
print(result.data_graph.serialize(format="turtle12"))


@prefix ex: <http://example.org/> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

ex:alice ex:isPerson true ;
    a ex:Person .

ex:bob ex:isPerson true ;
    a ex:Person .



## 2. Node expressions inside rule templates

`sh:subject`/`sh:predicate`/`sh:object` aren't limited to constants like `sh:this` or a literal - any of the three can be a full SHACL 1.2 node expression (see the separate Node Expressions guide for the expression language itself). This is the same mechanism either way; `sh:this` is just the simplest possible expression.

**Example 2.1** computes the object: `ex:skillCount` is derived via `shnex:count [ shnex:pathValues ex:skill ]`, the same `shnex:` vocabulary used throughout the Node Expressions guide - not a rules-specific mini-language.


**Example 2.1**

In [3]:
data = StarLayerGraph()
data.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:alice a ex:Person ; ex:skill ex:python, ex:sparql, ex:rdf .
""", format="turtle12")

shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    @prefix shnex: <http://www.w3.org/ns/shacl-node-expr#> .
    ex:S a sh:NodeShape ; sh:targetClass ex:Person ;
      sh:rule [
        a sh:TripleRule ;
        sh:subject sh:this ;
        sh:predicate ex:skillCount ;
        sh:object [ shnex:count [ shnex:pathValues ex:skill ] ] ;
      ] .
""", format="turtle12")

result = StarShaclValidator().apply_rules(data_graph=data, shacl_graph=shapes)
print(result.data_graph.serialize(format="turtle12"))


@prefix ex: <http://example.org/> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

ex:alice ex:skill ex:python, ex:sparql, ex:rdf ;
    ex:skillCount 3 ;
    a ex:Person .



**Example 2.2** goes further and computes the *subject* instead - the rule fires per `ex:Person`, but the inferred triple lands on that person's `ex:manager`, not on the person themselves. This works because a node expression's result, not the focus node, becomes the actual triple subject.

**Example 2.2**

In [4]:
data = StarLayerGraph()
data.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:alice a ex:Person ; ex:manager ex:carol .
""", format="turtle12")

shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    @prefix shnex: <http://www.w3.org/ns/shacl-node-expr#> .
    ex:S a sh:NodeShape ; sh:targetClass ex:Person ;
      sh:rule [
        a sh:TripleRule ;
        sh:subject [ shnex:pathValues ex:manager ] ;
        sh:predicate ex:hasReports ;
        sh:object true ;
      ] .
""", format="turtle12")

result = StarShaclValidator().apply_rules(data_graph=data, shacl_graph=shapes)
print(result.data_graph.serialize(format="turtle12"))
print("ex:carol's inferred triple, even though ex:alice was the target:", (EX.carol, EX.hasReports, None) in result.data_graph)


@prefix ex: <http://example.org/> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

ex:alice ex:manager ex:carol ;
    a ex:Person .

ex:carol ex:hasReports true .

ex:carol's inferred triple, even though ex:alice was the target: True


## 3. `sh:SPARQLRule`

A `sh:SPARQLRule` builds triples from a `sh:construct` CONSTRUCT query instead of subject/predicate/object templates - useful whenever the derived triple needs real computation (string building, aggregation, joins) that a bare template can't express. `$this` is pre-bound to the focus node inside the query.

Here, `ex:fullName` is built by concatenating `ex:firstName` and `ex:lastName`.

**Example 2**

In [5]:
data = StarLayerGraph()
data.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:alice a ex:Person ; ex:firstName "Alice" ; ex:lastName "Smith" .
""", format="turtle12")

shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    ex:FullNameRule a sh:NodeShape ; sh:targetClass ex:Person ;
      sh:rule [
        a sh:SPARQLRule ;
        sh:construct \"\"\"
          PREFIX ex: <http://example.org/>
          CONSTRUCT { $this ex:fullName ?full }
          WHERE { $this ex:firstName ?f ; ex:lastName ?l . BIND(CONCAT(?f, \" \", ?l) AS ?full) }
        \"\"\" ;
      ] .
""", format="turtle12")

result = StarShaclValidator().apply_rules(data_graph=data, shacl_graph=shapes)
print(result.data_graph.serialize(format="turtle12"))


@prefix ex: <http://example.org/> .

ex:alice ex:firstName "Alice" ;
    ex:fullName "Alice Smith" ;
    ex:lastName "Smith" ;
    a ex:Person .


## 4. `sh:condition`

`sh:condition` gates which focus nodes a rule fires for: only nodes that conform to the condition shape become focus nodes for that rule. The condition shape needs to actually *constrain* something (`sh:class`, `sh:property`, etc.) - a shape with only `sh:targetClass` trivially conforms for any node when checked this way, since targets only matter for `validate()`'s own target resolution, not an ad hoc conformance check.

Here, only `ex:alice` (an `ex:Adult`) gets the inferred voting-eligibility triple; `ex:bob` (an `ex:Minor`) doesn't.

**Example 3**

In [6]:
data = StarLayerGraph()
data.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:alice a ex:Adult ; ex:name "Alice" .
    ex:bob a ex:Minor ; ex:name "Bob" .
""", format="turtle12")

shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    ex:AdultShape a sh:NodeShape ; sh:class ex:Adult .
    ex:VotingRule a sh:NodeShape ; sh:targetSubjectsOf ex:name ;
      sh:rule [
        a sh:TripleRule ;
        sh:condition ex:AdultShape ;
        sh:subject sh:this ; sh:predicate ex:eligibleForVoting ; sh:object true ;
      ] .
""", format="turtle12")

result = StarShaclValidator().apply_rules(data_graph=data, shacl_graph=shapes)
print(result.data_graph.serialize(format="turtle12"))


@prefix ex: <http://example.org/> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

ex:alice ex:eligibleForVoting true ;
    ex:name "Alice" ;
    a ex:Adult .

ex:bob ex:name "Bob" ;
    a ex:Minor .


## 5. Rules run to a fixpoint by default

`apply_rules()` re-runs a shape's rules repeatedly until nothing new is produced (`iterate_rules=True`, the default) - not just once. This matters for anything transitive: a single pass only sees one hop, while the fixpoint sees the full closure.

Here, `ex:reach` is built by joining `ex:reach` with itself - a single pass only derives `a -> c` (one extra hop beyond what's stored); the fixpoint also derives `a -> d`, since the second pass can use the first pass's own output.

**Example 4**

In [7]:
data = StarLayerGraph()
data.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:a ex:reach ex:b .
    ex:b ex:reach ex:c .
    ex:c ex:reach ex:d .
""", format="turtle12")

shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    ex:ReachRule a sh:NodeShape ; sh:targetSubjectsOf ex:reach ;
      sh:rule [
        a sh:SPARQLRule ;
        sh:construct \"\"\"
          PREFIX ex: <http://example.org/>
          CONSTRUCT { $this ex:reach ?z }
          WHERE { $this ex:reach ?y . ?y ex:reach ?z }
        \"\"\" ;
      ] .
""", format="turtle12")

#a fresh copy for the single-pass run, so the fixpoint run below starts from
#the same original data rather than the single-pass result.
single_pass_data = StarLayerGraph()
single_pass_data.parse(data=data.serialize(format="turtle12"), format="turtle12")
single_pass = StarShaclValidator().apply_rules(data_graph=single_pass_data, shacl_graph=shapes, iterate_rules=False)
print("single pass, ex:a reaches:", sorted(str(o) for o in single_pass.data_graph.objects(EX.a, EX.reach)))

fixpoint = StarShaclValidator().apply_rules(data_graph=data, shacl_graph=shapes)
print("fixpoint,    ex:a reaches:", sorted(str(o) for o in fixpoint.data_graph.objects(EX.a, EX.reach)))


single pass, ex:a reaches: ['http://example.org/b', 'http://example.org/c']


fixpoint,    ex:a reaches: ['http://example.org/b', 'http://example.org/c', 'http://example.org/d']


## 6. Ordering: `sh:layer` and `sh:runOnce`

By default, each shape's rules reach their own fixpoint independently, shape by shape. `sh:layer` (an integer, default 0) groups rules across *every* shape into ordered layers - all of layer 0 runs (to its own shared fixpoint) before layer 1 starts, so a later layer can depend on an earlier layer's complete output even across shapes.

`sh:runOnce` marks a rule to fire at most once per focus node within its layer, rather than iterating to a fixpoint - essential for a rule that mints something new (like a fresh blank node) on every execution, which would otherwise look "new" forever and hit the iteration cap (section 12).

**Example 5.1**

In [8]:
#layer 1's rule targets whatever layer 0 produced - only works because
#layer 0 has already reached completion by the time layer 1 starts.
data = StarLayerGraph()
data.parse(data="@prefix ex: <http://example.org/> . ex:alice a ex:Person .", format="turtle12")

shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    ex:Layer0Rule a sh:NodeShape ; sh:targetClass ex:Person ;
      sh:rule [
        a sh:TripleRule ; sh:layer 0 ;
        sh:subject sh:this ; sh:predicate ex:stage ; sh:object "layer0" ;
      ] .
    ex:Layer1Rule a sh:NodeShape ; sh:targetSubjectsOf ex:stage ;
      sh:rule [
        a sh:TripleRule ; sh:layer 1 ;
        sh:subject sh:this ; sh:predicate ex:stage2 ; sh:object "layer1-saw-stage" ;
      ] .
""", format="turtle12")

result = StarShaclValidator().apply_rules(data_graph=data, shacl_graph=shapes)
print(result.data_graph.serialize(format="turtle12"))


@prefix ex: <http://example.org/> .

ex:alice ex:stage "layer0" ;
    ex:stage2 "layer1-saw-stage" ;
    a ex:Person .



**Example 5.2**

In [9]:
#without sh:runOnce, a rule minting a fresh blank node every execution would
#never converge (each pass looks "new") - sh:runOnce fires it exactly once.
data = StarLayerGraph()
data.parse(data="@prefix ex: <http://example.org/> . ex:alice a ex:Person .", format="turtle12")

shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    ex:MintRule a sh:NodeShape ; sh:targetClass ex:Person ;
      sh:rule [
        a sh:SPARQLRule ; sh:runOnce true ;
        sh:construct \"\"\"
          PREFIX ex: <http://example.org/>
          CONSTRUCT { $this ex:tag [ a ex:Tag ] }
          WHERE { }
        \"\"\" ;
      ] .
""", format="turtle12")

result = StarShaclValidator().apply_rules(data_graph=data, shacl_graph=shapes)
print("ex:tag count (expect 1, not an iteration-cap crash):", len(list(result.data_graph.objects(EX.alice, EX.tag))))


ex:tag count (expect 1, not an iteration-cap crash): 1


## 7. Rule sets: running only some rules

`sh:RuleSet`/`sh:hasRule`/`sh:includesRuleSet` names a group of rules (with transitive inclusion of other rule sets). `apply_rules(..., rule_set=<IRI>)` restricts execution to that set's closure - every other rule in the shapes graph is simply not run, regardless of which shape it's attached to.

Here, two shapes each contribute one rule; restricting to `ex:GreetingsOnly` runs only `ex:GreetingRule`, leaving `ex:farewell` un-inferred even though `ex:FarewellRule`'s own shape still targets `ex:alice`.

**Example 6**

In [10]:
data = StarLayerGraph()
data.parse(data="@prefix ex: <http://example.org/> . ex:alice a ex:Person .", format="turtle12")

shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    ex:R1 a sh:NodeShape ; sh:targetClass ex:Person ; sh:rule ex:GreetingRule .
    ex:GreetingRule a sh:TripleRule ;
      sh:subject sh:this ; sh:predicate ex:greeting ; sh:object "hi" .

    ex:R2 a sh:NodeShape ; sh:targetClass ex:Person ; sh:rule ex:FarewellRule .
    ex:FarewellRule a sh:TripleRule ;
      sh:subject sh:this ; sh:predicate ex:farewell ; sh:object "bye" .

    ex:GreetingsOnly a sh:RuleSet ; sh:hasRule ex:GreetingRule .
""", format="turtle12")

result = StarShaclValidator().apply_rules(data_graph=data, shacl_graph=shapes, rule_set=EX.GreetingsOnly)
print(result.data_graph.serialize(format="turtle12"))


@prefix ex: <http://example.org/> .

ex:alice ex:greeting "hi" ;
    a ex:Person .


## 8. Global rules: not attached to any shape

A `sh:SPARQLRule` normally needs a shape's `sh:rule` to reach it - but a bare `sh:SPARQLRule` node with no incoming `sh:rule` edge from any shape still runs, as a *global* rule with no focus node at all (`$this` isn't bound; the query matches against the whole data graph directly). Useful for a fact that isn't naturally about one particular node - here, the symmetric closure of any predicate typed `ex:SymmetricProperty`.

(This global path is deliberately scoped to `sh:SPARQLRule` only - there's no defined meaning for a `sh:TripleRule`'s subject/predicate/object templates with no focus node to evaluate `sh:this` against.)

**Example 7**

In [11]:
data = StarLayerGraph()
data.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:siblingOf a ex:SymmetricProperty .
    ex:alice ex:siblingOf ex:bob .
""", format="turtle12")

shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    ex:SymmetricPropertyRule a sh:SPARQLRule ;
      sh:construct \"\"\"
        PREFIX ex: <http://example.org/>
        CONSTRUCT { ?o ?p ?s . }
        WHERE { ?p a ex:SymmetricProperty . ?s ?p ?o . }
      \"\"\" .
""", format="turtle12")

result = StarShaclValidator().apply_rules(data_graph=data, shacl_graph=shapes)
print(result.data_graph.serialize(format="turtle12"))


@prefix ex: <http://example.org/> .

ex:alice ex:siblingOf ex:bob .

ex:bob ex:siblingOf ex:alice .

ex:siblingOf a ex:SymmetricProperty .



## 9. Provenance: `sh:sourceRule`

`apply_rules(..., include_source_rule_provenance=True)` attributes every inferred triple to the exact rule that produced it, via an RDF 1.2 reifier: `<< s p o >> {| sh:sourceRule <rule> |}`. Off by default (adds nothing). Useful when more than one rule could plausibly explain the same fact, or for auditing an inference pipeline.

The provenance itself is only materialized after *all* rule execution has fully finished - a rule's own WHERE clause can never see a `sh:sourceRule` triple appear mid-run, even for its own output.

**Example 8**

In [12]:
data = StarLayerGraph()
data.parse(data="@prefix ex: <http://example.org/> . ex:alice a ex:Person .", format="turtle12")

shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    ex:R a sh:NodeShape ; sh:targetClass ex:Person ; sh:rule ex:GreetingRule .
    ex:GreetingRule a sh:TripleRule ;
      sh:subject sh:this ; sh:predicate ex:greeting ; sh:object "hi" .
""", format="turtle12")

result = StarShaclValidator().apply_rules(data_graph=data, shacl_graph=shapes, include_source_rule_provenance=True)
print(result.data_graph.serialize(format="turtle12"))


@version "1.2" .
@prefix ex: <http://example.org/> .
@prefix sh: <http://www.w3.org/ns/shacl#> .

ex:alice ex:greeting "hi" {| sh:sourceRule ex:GreetingRule |} ;
    a ex:Person .


## 10. Rules and RDF 1.2: triple terms

A `sh:SPARQLRule`'s CONSTRUCT can build an RDF 1.2 triple term directly - not just plain triples. Here, alongside the ordinary `ex:knows` fact, the rule also builds a `<<( ... )>>` triple term "witness" of that same fact, evaluated via `starlayergraph`'s native RDF-1.2-aware query engine.

**Example 9**

In [13]:
data = StarLayerGraph()
data.parse(data="@prefix ex: <http://example.org/> . ex:alice ex:knows ex:bob .", format="turtle12")

shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    ex:R a sh:NodeShape ; sh:targetSubjectsOf ex:knows ;
      sh:rule [
        a sh:SPARQLRule ;
        sh:construct \"\"\"
          PREFIX ex: <http://example.org/>
          CONSTRUCT { $this ex:knowsWitness <<( $this ex:knows ?o )>> . }
          WHERE { $this ex:knows ?o . }
        \"\"\" ;
      ] .
""", format="turtle12")

result = StarShaclValidator().apply_rules(data_graph=data, shacl_graph=shapes)
print(result.data_graph.serialize(format="turtle12"))


@version "1.2" .
@prefix ex: <http://example.org/> .

ex:alice ex:knows ex:bob ;
    ex:knowsWitness <<( ex:alice ex:knows ex:bob )>> .



## 11. `apply_rules()` requires a `StarLayerGraph`, and always mutates it

`apply_rules()` runs with `inplace=True` specifically so `result.data_graph` is your own object, mutated with the inferred triples - not a disconnected copy you have to remember to swap in. That guarantee only makes sense for a `StarLayerGraph`: a plain `rdflib.Graph` can't hold StarLayerGraph's own RDF-1.2 encoding, so silently accepting one would mean building a *new*, separate `StarLayerGraph` internally and mutating *that* - `result.data_graph` would quietly stop being the object you passed in, with nothing telling you so.

Rather than let that divergence pass quietly, `apply_rules()` requires `data_graph` to already be a `StarLayerGraph` and raises `TypeError` otherwise - convert first (e.g. `StarLayerGraph.from_rdflib(g)`) if you're starting from a plain `rdflib.Graph`. One input type, one behavior: `data_graph` is always mutated in place, and `result.data_graph is data_graph` is always `True`. (`shacl_graph`, by contrast, is never mutated in place either way - `apply_rules()` always works from a copy of it.)

**Example 11**

In [14]:
shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    ex:R a sh:NodeShape ; sh:targetClass ex:Person ;
      sh:rule [ a sh:TripleRule ; sh:subject sh:this ; sh:predicate ex:tag ; sh:object "x" ] .
""", format="turtle12")

#a StarLayerGraph - required, and mutated in place
sl = StarLayerGraph()
sl.parse(data="@prefix ex: <http://example.org/> . ex:alice a ex:Person .", format="turtle12")
result = StarShaclValidator().apply_rules(data_graph=sl, shacl_graph=shapes)
print("result.data_graph is the same object you passed in:", result.data_graph is sl)

#a plain rdflib.Graph - rejected outright, rather than silently returning
#a disconnected copy that only looks like it mutated your original object.
import rdflib
plain = rdflib.Graph()
plain.parse(data="@prefix ex: <http://example.org/> . ex:alice a ex:Person .", format="turtle")
try:
    StarShaclValidator().apply_rules(data_graph=plain, shacl_graph=shapes)
except TypeError as e:
    print("raised as expected:", e)


result.data_graph is the same object you passed in:

 True


raised as expected: apply_rules() requires data_graph to be a StarLayerGraph, since rule execution mutates it in place - got Graph. Convert first, e.g. StarLayerGraph.from_rdflib(data_graph).


## 12. Known limits

**The iteration cap is real, not silently truncated.** pySHACL guards against runaway rule loops with a default 100-iteration cap; exceeding it raises `pyshacl.errors.ReportableRuntimeError`, not a silently-incomplete result. Demonstrated below with the cap lowered to make it reachable quickly (the actual default of 100 would take real chains far longer to hit).

**`sh:PropertyRule`/`sh:values`** (a `sh:rule` shorthand - fixed `sh:path`, values from a node expression, no explicit subject/predicate/object template) is **not implemented**, in pySHACL or in starshacl - using it raises `pyshacl.errors.RuleLoadError` rather than silently no-opping. `sh:TripleRule` and `sh:SPARQLRule` remain the two supported rule types.

**Example 11.1**

In [15]:
import pyshacl.rules
import pyshacl.rules.sparql
from pyshacl.errors import ReportableRuntimeError

data = StarLayerGraph()
data.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:a ex:reach ex:b . ex:b ex:reach ex:c . ex:c ex:reach ex:d .
    ex:d ex:reach ex:e . ex:e ex:reach ex:f . ex:f ex:reach ex:g .
""", format="turtle12")

shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    ex:ReachRule a sh:NodeShape ; sh:targetSubjectsOf ex:reach ;
      sh:rule [
        a sh:SPARQLRule ;
        sh:construct \"\"\"
          PREFIX ex: <http://example.org/>
          CONSTRUCT { $this ex:reach ?z }
          WHERE { $this ex:reach ?y . ?y ex:reach ?z }
        \"\"\" ;
      ] .
""", format="turtle12")

#lower the cap just to make it reachable in this example - restore it after.
original_limit = pyshacl.rules.RULES_ITERATE_LIMIT
original_sparql_limit = pyshacl.rules.sparql.SPARQL_RULE_ITERATE_LIMIT
pyshacl.rules.RULES_ITERATE_LIMIT = 3
pyshacl.rules.sparql.SPARQL_RULE_ITERATE_LIMIT = 3
try:
    StarShaclValidator().apply_rules(data_graph=data, shacl_graph=shapes)
except ReportableRuntimeError as e:
    print("raised as expected:", e)
finally:
    pyshacl.rules.RULES_ITERATE_LIMIT = original_limit
    pyshacl.rules.sparql.SPARQL_RULE_ITERATE_LIMIT = original_sparql_limit


raised as expected: SHACL Shape Rule iteration exceeded iteration limit of 3.


**Example 11.2**

In [16]:
from pyshacl.errors import RuleLoadError

data = StarLayerGraph()
data.parse(data='@prefix ex: <http://example.org/> . ex:alice a ex:Person ; ex:name "Alice" .', format="turtle12")

shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    ex:R a sh:NodeShape ; sh:targetClass ex:Person ;
      sh:rule [ a sh:PropertyRule ; sh:path ex:fullName ; sh:values ex:name ] .
""", format="turtle12")

try:
    StarShaclValidator().apply_rules(data_graph=data, shacl_graph=shapes, meta_shacl=False)
except RuleLoadError as e:
    print("raised as expected:", e)


raised as expected: when using sh:rule, the Rule must be defined as either a TripleRule or SPARQLRule.
For reference, see https://www.w3.org/TR/shacl-af/#rules-syntax


## Further work

- **`sh:tempTriple`** (a triple visible only during rule execution, stripped before the final result) isn't demonstrated here - it's supported for shape-attached rules but not yet wired into the global-rules path (section 8).
- **`sh:expectedPredicate`** (ensuring a `sh:defaultValue`/`sh:values`-derived property exists before a rule runs) isn't demonstrated here - only the `sh:defaultValue` half is implemented; the `sh:values` half depends on the still-unimplemented `sh:PropertyRule`/`sh:values` gap above.
- **Cross-path layering.** The shape-attached layered loop (section 6) and the global-rules layered loop (section 8) are two separate fixpoints, not one unified cross-path fixpoint - a global `sh:runOnce` rule creating instances a shape-attached rule needs as *implicit-class* targets isn't picked up dynamically (targets are computed once, early, before any rule runs).
- **`sh:RuleSet` composition with provenance/layering** works (only the rules actually run get `sh:sourceRule` provenance, and rule sets honor `sh:layer`/`sh:runOnce` normally) but isn't separately demonstrated here.
- **The spec citation is still settling.** As of this writing, the rules vocabulary's home (`SHACL 1.2 Inference Rules`) is an editor's draft with no published `/TR/` snapshot - it moved twice in less than a month before landing here. See `docs/shacl12-gap-matrix.md`'s "Tracking Upstream Spec Changes" section before citing a specific section number externally.